# Jour 3 · Introduction au Deep Learning et à l'autoencoder


## Objectifs

- comprendre entrée, couche Dense, poids, loss et entraînement
- entraîner un autoencoder uniquement sur le comportement normal
- détecter avec l'erreur de reconstruction

## Le minimum nécessaire

Un neurone calcule une combinaison pondérée de ses entrées puis applique éventuellement une activation. Une couche Dense regroupe plusieurs neurones. Pendant l'entraînement, l'optimiseur ajuste les poids pour réduire une **loss**.

Le petit exemple suivant apprend approximativement la relation `y = 2x + 1`. Il ne s'agit pas encore de série temporelle : il sert à voir le mécanisme.

![Entrées pondérées, neurone, sortie et boucle de réduction de la loss](../assets/jour_03/03_neurone_poids_loss.png)

*L'entraînement ajuste progressivement les poids afin de réduire l'écart entre la sortie produite et la valeur attendue.*

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")
print("TensorFlow", tf.__version__)

In [ ]:
x = np.linspace(-1, 1, 200).reshape(-1, 1).astype("float32")
y = (2 * x + 1 + np.random.default_rng(42).normal(0, 0.08, x.shape)).astype("float32")

neuron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(1),
])
neuron.compile(optimizer="adam", loss="mse")
history = neuron.fit(x, y, epochs=60, verbose=0)

print("Prédiction pour x=3 :", float(neuron.predict(np.array([[3.0]], dtype="float32"), verbose=0)[0, 0]))
plt.plot(history.history["loss"])
plt.xlabel("époque")
plt.ylabel("loss (MSE)")
plt.title("La loss diminue pendant l'entraînement")
plt.show()

## Principe de l'autoencoder

`mesures → encoder → bottleneck → decoder → mesures reconstruites`

Il apprend à recopier les comportements normaux en passant par une représentation comprimée. Une observation très différente est souvent moins bien reconstruite. Son erreur de reconstruction devient notre score d'anomalie.

![Architecture d'un autoencoder avec encoder, bottleneck, decoder et reconstruction](../assets/jour_03/03_architecture_autoencoder.png)

*Le bottleneck force le réseau à conserver une représentation compacte du comportement normal avant de reconstruire les mesures.*

![Différence entre entrée et reconstruction puis distribution des erreurs avec un seuil](../assets/jour_03/03_erreur_reconstruction_seuil.png)

*Le seuil est choisi à partir des erreurs normales d'entraînement ; les reconstructions très difficiles deviennent des anomalies possibles.*

In [ ]:
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_labeled.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.set_index("timestamp").sort_index()
features = ["temperature_c", "humidity_pct", "power_kw", "pressure_bar", "vibration_mm_s"]

train_end = df.index.min() + pd.Timedelta(days=28)
normal_train = df.loc[(df.index < train_end) & (df["is_anomaly"] == 0), features]

scaler = StandardScaler()
x_train = scaler.fit_transform(normal_train).astype("float32")
x_all = scaler.transform(df[features]).astype("float32")
print("Exemples normaux pour apprendre :", len(x_train))

In [ ]:
autoencoder = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(3, activation="relu", name="encoder"),
    tf.keras.layers.Dense(2, activation="relu", name="bottleneck"),
    tf.keras.layers.Dense(3, activation="relu", name="decoder"),
    tf.keras.layers.Dense(len(features), name="reconstruction"),
])
autoencoder.compile(optimizer="adam", loss="mse")
history = autoencoder.fit(
    x_train,
    x_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=0,
)

pd.DataFrame(history.history).plot(figsize=(8, 3), title="Apprentissage de la reconstruction")
plt.xlabel("époque")
plt.ylabel("MSE")
plt.show()

In [ ]:
reconstructed = autoencoder.predict(x_all, verbose=0)
df["reconstruction_error"] = np.mean((x_all - reconstructed) ** 2, axis=1)

train_errors = df.loc[normal_train.index, "reconstruction_error"]
threshold = train_errors.quantile(0.99)
df["autoencoder_flag"] = (df["reconstruction_error"] > threshold).astype(int)

test = df.loc[df.index >= train_end]
print("Seuil :", round(threshold, 4))
print("Précision :", round(precision_score(test["is_anomaly"], test["autoencoder_flag"], zero_division=0), 3))
print("Rappel    :", round(recall_score(test["is_anomaly"], test["autoencoder_flag"], zero_division=0), 3))
print("F1        :", round(f1_score(test["is_anomaly"], test["autoencoder_flag"], zero_division=0), 3))

In [ ]:
view = test.last("14D")
flagged = view["autoencoder_flag"] == 1
ax = view["reconstruction_error"].plot(figsize=(13, 4), label="erreur de reconstruction")
ax.axhline(threshold, color="red", linestyle="--", label="seuil")
ax.scatter(view.index[flagged], view.loc[flagged, "reconstruction_error"], color="red", s=12)
ax.set_title("Une reconstruction difficile devient une alerte")
ax.legend()
plt.show()

### À vous de jouer — observer le compromis du seuil

Comparez les quantiles 0,98, 0,99 et 0,995 des erreurs normales d'entraînement. Affichez nombre d'alertes, précision et rappel sur le test.

In [ ]:
# Écrivez votre code ici.
pass

## Limites à comprendre

Cet autoencoder Dense traite chaque ligne séparément : il ne comprend pas encore une séquence. Il peut néanmoins détecter des combinaisons inhabituelles. En production, le scaler et le réseau doivent être versionnés ensemble, et le seuil doit être surveillé.